# Phase 2.1 end-to-end workflow

This notebook is the reproducible top-level caller for the independent Phase 2.1 package. It imports the public `Phase21Config`, `run_phase21`, and `save_phase21_result` interfaces, then optionally creates all requested figures.

This notebook always uses the real TNG50-1 inputs and the eight published Henriques cooling tables.

In [ ]:
from pathlib import Path
import json
import os
import sys

def locate_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'src' / 'analysis' / 'pipeline.py').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook from project/2.1 or its parent directory.')

PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT={PROJECT_ROOT}')

In [ ]:
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import Phase21Config, run_phase21, save_phase21_result
from src.physics.cooling_function import LgalCoolingFunction

print({
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'h5py': h5py.__version__,
    'matplotlib': matplotlib.__version__,
})

REBUILD_SAMPLE = False
STATE_WORKERS = int(os.environ.get('PHASE21_STATE_WORKERS', '1'))
STATE_BACKEND = os.environ.get('PHASE21_STATE_BACKEND', 'thread')

BASE_PATH = Path(os.environ.get('TNG50_BASE_PATH', Phase21Config.base_path))
COOLING_TABLE_DIR = PROJECT_ROOT / 'data' / 'external' / 'cooling_tables'
CACHE_DIR = PROJECT_ROOT / 'data' / 'interim' / 'cache'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'results' / 'figures'
config = Phase21Config(
    base_path=str(BASE_PATH),
    state_workers=STATE_WORKERS,
    state_parallel_backend=STATE_BACKEND,
)
print(f'base_path={config.base_path}')
print(f'state_workers={config.state_workers}, backend={config.state_parallel_backend}')
print(f'fingerprint={config.fingerprint}')

## Full Phase 2.1 call

The following cell is the notebook equivalent of `scripts/run_analysis.py`: it loads the cooling law, scans the sample and snap90--99 history, classifies snap94 → 95 events including `other`, computes statistics, serializes the result tree, and returns paths for every saved product.

In [ ]:
cooling_function = LgalCoolingFunction.from_directory(COOLING_TABLE_DIR)
result = None
saved_paths = {}
result = run_phase21(
    cooling_function,
    config=config,
    cache_dir=CACHE_DIR,
    rebuild_sample=REBUILD_SAMPLE,
    verbose=True,
)
output_prefix = OUTPUT_DIR / 'phase21_snap090_099_seed202608'
saved_paths = save_phase21_result(result, output_prefix)
for label, path in saved_paths.items():
    print(f'{label}: {path}')

In [ ]:
if result is not None:
    from src.plotting import plot_agn_cooling, plot_composition, plot_feedback_before_accretion

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    plot_agn_cooling(
        result['agn_quartile_statistics'],
        halo_results=result['halo_results'],
        save_path=FIGURE_DIR / 'phase21_mdot_heat_h15_quartile_cooling.png',
    )
    plot_composition(
        result['composition_statistics'],
        save_path=FIGURE_DIR / 'phase21_composition.png',
    )
    plot_feedback_before_accretion(
        result['normalized_statistics'],
        save_path=FIGURE_DIR / 'phase21_feedback_before_accretion.png',
    )
    print(f'figures written to {FIGURE_DIR}')
else:
    print('Run the full Phase 2.1 call cell before plotting.')

## Closure and output audit

In [ ]:
if result is not None:
    metadata = result['metadata']
    closure = result['closure_diagnostics']
    composition = result['composition_statistics']
    print('selected halos:', metadata['selected_halo_count'])
    print('fingerprint:', metadata['fingerprint'])
    print('closure:', json.dumps(closure, indent=2))
    print('entry component names:', composition['in_component_names'].tolist())
    print('exit component names:', composition['out_component_names'].tolist())
    assert closure['max_abs_count_in_error'] == 0
    assert closure['max_abs_count_out_error'] == 0
else:
    print('No result to audit; run the full-call cell first.')

## Command-line equivalent

From the `project/2.1` directory, the equivalent full run is:

```bash
python scripts/run_analysis.py --base-path /path/to/TNG50-1/output --cooling-table-dir data/external/cooling_tables --cache-dir data/interim/cache --output-dir data/processed --figure-dir results/figures
```